# Reinforcement Learning with Verifiable Rewards (RLVR)

In Session 15 we changed a model's weights with GRPO, rewarding it for verifiably correct answers. This session zooms into the other half of that loop — the **verifier** — and the data pipeline built around it. No GPU required: we run the RLVR sampling-and-verification loop against an API model, so the focus stays on the part that makes or breaks reinforcement learning on language models: the reward signal itself.

The RLVR loop looks like this:

```text
prompt -> sample N completions -> verify each against a deterministic checker
       -> assign rewards -> keep verified-correct samples as preference data
       -> policy update -> repeat
```

Unlike RLHF, there is no learned reward model and no human labeler in the loop. The reward comes from a *deterministic program* — a math answer checker, a unit-test runner — that either passes a completion or doesn't. That makes the signal cheap, objective, and reproducible. It also makes it a target: any policy trained against a verifier will find and exploit its blind spots, so we will also build reward-hacking detection and an audit trail that records every verifier decision.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain how RLVR differs from RLHF, and what makes a reward "verifiable."
- Implement verifiable reward functions for math (exact-answer matching) and code (unit-test execution).
- Run the sample-and-verify loop and interpret group-level accuracy.
- Detect reward-hacking signatures and maintain a verifier audit trail.
- Construct chosen/rejected preference pairs ready for DPO-style training — or reward signals ready for GRPO.

## Table of Contents

- **Breakout Room #1: The Verifiable Reward Loop**
  - Task 1: Environment Setup
  - Task 2: Problems and Answer Extraction
  - Task 3: A Math Reward Function
  - Question #1 and Question #2
  - Task 4: Sample and Verify
- **Breakout Room #2: Reward Hacking, Code Verification, and Preference Data**
  - Task 5: Reward-Hacking Detection and the Audit Trail
  - Question #3
  - Task 6: A Code Verifier
  - Question #4
  - Task 7: Build Preference Pairs
  - Activity #1
- **Conclusion: What We Built, Start to Finish**
- **What Looks Different in Production**

---
# Breakout Room #1
## The Verifiable Reward Loop

We build the core RLVR machinery: a small set of math problems with known answers, a deterministic answer checker, a reward function, and the sample-and-verify loop that turns them into training signal.

## Task 1: Environment Setup

From the `16_RLVR` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

You will need an [OpenAI API key](https://platform.openai.com/api-keys). Enter it below — it is kept in memory for this session only.

In [2]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

### The Policy

In RL terms, the model we sample from is the **policy**. We deliberately use a small model (`gpt-4.1-nano`) at temperature 1.0: a policy that is *sometimes wrong* is exactly what we want, because the contrast between verified-correct and verified-incorrect samples is where the training signal lives. A policy that never fails produces no gradient — and no preference pairs.

In [3]:
from openai import OpenAI

client = OpenAI()

MODEL = "gpt-4.1-nano"  # small on purpose: we *want* some wrong answers


def simple_complete(prompt: str, system: str = "", temperature: float = 1.0) -> str:
    """One completion from the policy model."""
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


print(simple_complete("Reply with exactly: policy online"))

policy online


## Task 2: Problems and Answer Extraction

RLVR only works in domains where correctness can be *checked by a program*. Math word problems with a single numeric answer are the canonical example (this is why GSM8K shows up in every RLVR paper — and in Session 15).

Two conventions make checking reliable:

1. Each problem carries a **ground-truth answer** as a string.
2. The prompt instructs the policy to put its final answer in `\boxed{}` — the same convention GSM8K-style training uses — so extraction is a regex, not a judgment call. If no box is found, we fall back to the last number in the text.

In [4]:
import re
from dataclasses import dataclass


@dataclass
class Problem:
    question: str
    answer: str  # ground truth


problems = [
    Problem("What is 12 * 13?", "156"),
    Problem("If a train covers 60 km in 45 minutes, what is its speed in km/h?", "80"),
    Problem("What is the sum of the first 10 positive integers?", "55"),
    Problem(
        "A store discounts a $250 jacket by 20%, then adds 10% sales tax on the discounted price. "
        "What is the final price in dollars?",
        "220",
    ),
    Problem("How many positive divisors does 360 have?", "24"),
]


def extract_number(text: str) -> str:
    r"""Pull the final answer out of a completion: \boxed{...} first, last number as fallback."""
    boxed = re.search(r"\\boxed\{([^}]+)\}", text)
    if boxed:
        return boxed.group(1).strip()
    numbers = re.findall(r"-?\d+\.?\d*", text)
    return numbers[-1] if numbers else ""


assert extract_number(r"Step 1: 12*13 = 156. The answer is \boxed{156}.") == "156"
assert extract_number("So the speed is 80 km/h") == "80"
assert extract_number("no numbers here") == ""
print("extraction OK")

extraction OK


## Task 3: A Math Reward Function

The reward function is where verification becomes training signal:

- **+1.0** if the extracted answer matches the ground truth,
- **−0.1** otherwise.

Note the asymmetry: full credit for success, only a *small* penalty for failure. We also normalize numerically (`"80"` and `"80.0"` should match) — a verifier that fails on formatting technicalities punishes correct reasoning, which is the fastest way to teach a policy the wrong lesson.

In [5]:
class MathRewardFunction:
    """Verifiable reward for math problems: exact answer match against ground truth."""

    correct_reward = 1.0
    incorrect_penalty = -0.1

    def compute(self, response: str, ground_truth: str) -> float:
        prediction = extract_number(response)
        if self._normalize(prediction) == self._normalize(ground_truth):
            return self.correct_reward
        return self.incorrect_penalty

    @staticmethod
    def _normalize(value: str):
        """Compare numerically when possible, so '80' == '80.0'."""
        try:
            return float(value)
        except ValueError:
            return value.strip()


reward_fn = MathRewardFunction()

assert reward_fn.compute(r"The speed is \boxed{80}", "80.0") == 1.0
assert reward_fn.compute(r"The speed is \boxed{81}", "80") == -0.1
assert reward_fn.compute("I cannot solve this.", "80") == -0.1
print("reward function OK")

reward function OK


#### ❓ Question #1

In your own words: what makes a reward "verifiable," and how does RLVR differ from RLHF's learned reward model? Give one task where a verifiable reward exists and one where it fundamentally cannot (and explain why).

##### Answer:

A reward is **verifiable** when correctness can be decided by a *deterministic program* against a known ground truth — the same input always yields the same score, no learned model or human judgment is in the loop. The check is objective (right/wrong is defined by a checker, not an opinion), cheap (just run the program), and reproducible (anyone re-running the verifier gets the identical decision). In this notebook, `extract_number` + exact match on `\boxed{}` is such a program: given a completion and a ground-truth string, the reward is fully determined.

**RLVR vs. RLHF.** RLHF trains a *learned reward model* on human preference labels, then optimizes the policy against that model. The reward signal is a neural net — it's an approximation, it can be noisy or biased, and it is itself hackable (the policy can find inputs the reward model scores highly but humans wouldn't). RLVR replaces that learned model with a fixed program, so:
- the reward is exact rather than approximate (no reward-model generalization error),
- there is no human labeling cost per example and no reward-model training step, and
- the signal is stable over the whole run — it doesn't drift as you'd have to worry about with a reward model that can go stale relative to the policy.

The trade-off: RLVR only applies where a checker exists. RLHF works anywhere you can collect preferences.

**A task where a verifiable reward exists:** solving a math problem with a single canonical answer (e.g., GSM8K). A program compares the extracted answer to the ground truth — pass/fail is objective. (Unit-tested code, in Task 6, is the same idea graded fractionally.)

**A task where it fundamentally cannot:** open-ended writing quality — e.g., "write a moving, elegant short story" or "make this email sound warmer." There is no ground-truth string and no deterministic program that decides whether prose is *good*; quality is subjective, multi-dimensional, and context-dependent. Any "checker" you wrote (length, keyword presence, a sentiment classifier) would be a proxy the policy would immediately hack, and it would not actually measure the thing you care about. These tasks need human preference (RLHF) or an LLM-judge precisely because correctness isn't programmatically decidable.

#### ❓ Question #2

The reward is asymmetric: +1.0 for a correct answer but only −0.1 for an incorrect one. Suppose we used −1.0 instead. What behavior might a policy learn during early training, when most of its attempts fail? (Hint: think about a model that discovers it can hedge, refuse, or produce no parseable answer at all.)

##### Answer:

Early in training the policy is *mostly wrong*, so with a symmetric −1.0 penalty the vast majority of samples return −1.0 and the expected reward of "actually attempting a hard problem" is strongly negative. The gradient then pushes the policy toward whatever minimizes losses rather than toward correct reasoning — and the easiest way to stop losing points is to **stop making checkable claims**:

- **Refuse or hedge.** "I can't determine this," "it depends," "there isn't enough information" — an answer with no committed number can't be scored wrong, so the policy learns refusal is safer than a genuine attempt.
- **Produce no parseable answer.** Omit the `\boxed{}`, trail off, or bury the number so extraction fails or grabs the wrong token. If the verifier still charges −1.0 for unparseable output, the model may go further and emit vague non-numeric text.
- **Collapse in diversity / mode-collapse onto "safe" filler.** Because the downside (−1.0) is as large as the upside (+1.0) but failure is far more *frequent* early on, exploration is discouraged; the policy under-attempts and never generates the correct/incorrect contrast that later learning depends on.

This is why the asymmetry matters. With **+1.0 / −0.1**, a correct answer is worth ~10× the cost of a wrong one, so *attempting* has positive expected value even at low accuracy (you only need to be right ~9% of the time to break even). The small penalty keeps a wrong-but-earnest attempt from being catastrophic, preserves exploration, and prevents the "learned refusal" collapse — while still nudging the policy to prefer being right. In short: a harsh symmetric penalty optimizes for *not being caught wrong*; the mild asymmetric penalty optimizes for *trying to be right*.

## Task 4: Sample and Verify

Now the heart of RLVR: for each problem, sample a **group** of completions at temperature 1.0 and verify every one.

Sampling groups (rather than one completion per prompt) is not incidental — it is the same structure GRPO consumed in Session 15, where each completion's advantage was computed *relative to its group's average reward*. Here the group serves a second purpose too: correct and incorrect completions of the same prompt become the raw material for preference pairs in Breakout Room #2.

In [6]:
from dataclasses import asdict


@dataclass
class Sample:
    problem: str
    response: str
    extracted: str
    reward: float
    verified_correct: bool


def sample_and_verify(problem: Problem, n_samples: int = 4) -> list[Sample]:
    """Sample a group of completions for one problem and verify each one."""
    group = []
    for _ in range(n_samples):
        response = simple_complete(
            f"Solve step by step. Put the final numeric answer in \\boxed{{}}.\n\n{problem.question}",
            system="You are a careful mathematician. Show your work, then box the final number.",
        )
        extracted = extract_number(response)
        reward = reward_fn.compute(response, problem.answer)
        group.append(
            Sample(
                problem=problem.question,
                response=response,
                extracted=extracted,
                reward=reward,
                verified_correct=reward > 0,
            )
        )
    return group


groups = [sample_and_verify(p) for p in problems]

total = sum(len(g) for g in groups)
correct = sum(s.verified_correct for g in groups for s in g)
print(f"Verified-correct rate: {correct / total:.0%} ({correct}/{total})\n")
for problem, group in zip(problems, groups):
    print(f"  {sum(s.verified_correct for s in group)}/{len(group)}  {problem.question[:70]}")

Verified-correct rate: 95% (19/20)

  4/4  What is 12 * 13?
  3/4  If a train covers 60 km in 45 minutes, what is its speed in km/h?
  4/4  What is the sum of the first 10 positive integers?
  4/4  A store discounts a $250 jacket by 20%, then adds 10% sales tax on the
  4/4  How many positive divisors does 360 have?


> NOTE: If your verified-correct rate is 100%, the contrast that drives learning is missing — swap in a smaller model, raise the temperature, or add harder problems until some samples fail.

Let's inspect one failure — reading verifier-rejected completions is how you learn what your policy actually gets wrong (arithmetic slips? misread units? unparseable formatting?):

In [7]:
incorrect = [s for g in groups for s in g if not s.verified_correct]
if incorrect:
    sample = incorrect[0]
    print(f"Problem:   {sample.problem}")
    print(f"Extracted: {sample.extracted!r}  (ground truth mismatch, reward {sample.reward})\n")
    print(sample.response)
else:
    print("No incorrect samples this run - try a harder problem or higher temperature.")

Problem:   If a train covers 60 km in 45 minutes, what is its speed in km/h?
Extracted: '80 \\text{ km/h'  (ground truth mismatch, reward -0.1)

Let's carefully go through the problem step by step.

1. **Identify the given information:**
   - Distance covered = 60 km
   - Time taken = 45 minutes

2. **Convert the time from minutes to hours:**
   Since 1 hour = 60 minutes, then:
   \[
   \text{Time in hours} = \frac{45}{60} = \frac{3}{4} \text{ hours}
   \]

3. **Calculate the speed in km/h:**
   Speed = Distance / Time
   \[
   \text{Speed} = \frac{60 \text{ km}}{\frac{3}{4} \text{ hr}} = 60 \times \frac{4}{3} = \frac{240}{3} = 80
   \]

**Final answer:**

\[
\boxed{80 \text{ km/h}}
\]


## Breakout Room #1 Summary

- Verifiable rewards come from deterministic checkers, not human preference or a learned reward model — cheap, objective, reproducible.
- The `\boxed{}` convention plus numeric normalization makes extraction mechanical; a brittle verifier punishes correct reasoning and corrupts the signal.
- Asymmetric rewards (+1.0 / −0.1) keep early training from collapsing into refusal.
- Sampling *groups* of completions per prompt is the same structure GRPO trains on — and it produces the correct/incorrect contrast that preference data needs.

---
# Breakout Room #2
## Reward Hacking, Code Verification, and Preference Data

A verifier is not just a metric — once you train against it, it *is* the objective. This room covers what happens then: policies that exploit the verifier's blind spots, a second verifiable domain (code judged by unit tests), and turning audited verifier output into preference data.

## Task 5: Reward-Hacking Detection and the Audit Trail

Goodhart's law — *"when a measure becomes a target, it ceases to be a good measure"* — is the central operational risk of RLVR. A policy optimized against an exact-match verifier will happily learn to:

- emit a bare boxed answer with no reasoning (guessing is cheap when only the box is checked),
- parrot numbers that appear in the prompt,
- or exploit extraction quirks instead of solving the problem.

Two defenses, both borrowed from how production RLVR systems are reviewed:

1. **Flag hack signatures.** Here we flag verified-correct samples that show *no visible work* — a right answer without reasoning is the classic signature of guessing or leakage.
2. **Log every verifier decision** to an append-only audit trail (`artifacts/verifier.jsonl`). When someone — a teammate, an auditor, a regulator — asks whether your RL run was trained on honest rewards, this file is the answer.

In [8]:
import json
from pathlib import Path

AUDIT_LOG = Path("artifacts/verifier.jsonl")
AUDIT_LOG.parent.mkdir(exist_ok=True)


def looks_like_hack(sample: Sample) -> bool:
    """Flag verified-correct samples that show no work.

    A correct boxed answer with no visible reasoning is the classic hack
    signature: the policy may be guessing, pattern-matching the prompt, or
    exploiting the extractor rather than solving the problem.
    """
    if not sample.verified_correct:
        return False
    work = sample.response.replace(f"\\boxed{{{sample.extracted}}}", "")
    numbers_in_work = re.findall(r"-?\d+\.?\d*", work)
    return len(sample.response.split()) < 20 or len(numbers_in_work) < 2


def audit_record(sample: Sample) -> dict:
    """Append one verifier decision to the audit trail and return it."""
    record = {**asdict(sample), "suspected_hack": looks_like_hack(sample)}
    with AUDIT_LOG.open("a") as f:
        f.write(json.dumps(record) + "\n")
    return record


records = [audit_record(s) for g in groups for s in g]
flagged = sum(r["suspected_hack"] for r in records)

print(f"Audited {len(records)} samples -> {flagged} flagged as hack-suspect")
print(f"Audit trail: {AUDIT_LOG} ({sum(1 for _ in AUDIT_LOG.open())} records total)")

Audited 20 samples -> 0 flagged as hack-suspect
Audit trail: artifacts/verifier.jsonl (20 records total)


#### ❓ Question #3

Our detector flags one signature: "right answer, no visible work." Name **two other ways** a policy could hack a `\boxed{}` exact-match verifier, and for each, describe how you would harden the verifier or the prompt against it. (Session 15's stacked format rewards are one relevant hardening example.)

##### Answer:

**1. Prompt-number parroting / spraying the box with candidates.**
The problem statement often *contains* the answer or numbers close to it (e.g., "discount a $250 jacket by 20%… final price?" — the ground truth 220 is derivable, but numbers like 250, 20, 10 are right there). A policy can learn to copy a salient prompt number into `\boxed{}`, or to emit *multiple* boxed values and let the extractor's "last box wins" rule catch the lucky one. Either way it scores without reasoning.
- **Hardening:** (a) In the reward function, penalize completions whose boxed answer is copied verbatim from the prompt when the true answer requires computation, and flag/zero-out completions containing *more than one* `\boxed{}` (the current extractor silently takes one — make multiple boxes an automatic reject). (b) Add a **process/format reward** (Session 15's stacked rewards): require the boxed number to be preceded by a derivation whose intermediate arithmetic is consistent with the final answer, so a bare parroted number earns the format penalty even when it matches. (c) Where feasible, use problems whose answer does **not** appear in the prompt.

**2. Extractor exploitation via formatting tricks.**
The regex `\\boxed\{([^}]+)\}` and the numeric-normalization step are attack surfaces. A policy can learn that `\boxed{80 \text{ km/h}}` fails but a bare `\boxed{80}` passes (we literally saw this failure), or exploit normalization: emit `80.0000001`, `+80`, `080`, `8e1`, or unicode look-alikes to slip through or around `float()` comparison, or nest braces to confuse `[^}]+`. More adversarially, it could append trailing text hoping the "last number" fallback grabs a value it prefers.
- **Hardening:** (a) Replace brittle exact/regex matching with a **symbolic/robust checker** — parse the boxed content and compare with `math.isclose` or SymPy equivalence, so `80`, `80.0`, `+80`, `80 km/h` all reduce to the same value *deliberately* rather than by accident. (b) Make the format **strict and rewarded**: require exactly one `\boxed{}` containing only a normalized number (via a stacked format reward), and reject completions that rely on the last-number fallback rather than an actual box, so the extractor's soft fallback can't be gamed. (c) Fuzz-test the verifier against adversarial formatting so blind spots are found by you before the policy finds them.

## Task 6: A Code Verifier

Math is one verifiable domain; **code judged by unit tests** is the other workhorse of RLVR. The verifier executes a candidate program against test cases and returns the *fraction that pass* — a graded reward in `[0.0, 1.0]` rather than math's binary match.

We run candidates in a subprocess with a timeout: a program that crashes, hangs, or exits non-zero simply earns no credit for that test case.

> ⚠️ We are executing model-generated code on your machine. For this demo the programs are trivial, but note the design: in production, this verifier runs inside a **sandbox** (container, gVisor, firecracker VM) — never on the host.

In [9]:
import subprocess
import sys


class CodeVerifier:
    """Score generated code by the fraction of test cases it passes."""

    timeout_seconds = 5

    def verify(self, code: str, test_cases: list[dict]) -> float:
        passed = 0
        for tc in test_cases:
            try:
                output = self._run(code, tc.get("input", ""))
                if output.strip() == str(tc["expected"]).strip():
                    passed += 1
            except Exception:
                pass  # crash, timeout, or non-zero exit -> no credit for this case
        return passed / len(test_cases) if test_cases else 0.0

    def _run(self, code: str, input_data: str = "") -> str:
        result = subprocess.run(
            [sys.executable, "-c", code],
            input=input_data,
            capture_output=True,
            text=True,
            timeout=self.timeout_seconds,
        )
        if result.returncode != 0:
            raise RuntimeError(result.stderr)
        return result.stdout


verifier = CodeVerifier()

# Sanity check with hand-written candidates: one correct, one buggy.
tests = [{"input": "3", "expected": "14"}, {"input": "10", "expected": "385"}]
good = "n = int(input()); print(sum(i * i for i in range(1, n + 1)))"
bad = "n = int(input()); print(sum(range(1, n + 1)))"  # sums i, not i^2

assert verifier.verify(good, tests) == 1.0
assert verifier.verify(bad, tests) == 0.0
print("code verifier OK")

code verifier OK


Now close the loop: have the **policy** write the program, and let the verifier score it — the exact reward signal a coding-RLVR run trains on.

In [10]:
CODING_TASK = (
    "Write a Python program that reads a single integer n from standard input "
    "and prints the sum of the squares of the integers from 1 to n (inclusive). "
    "Print only the number. Reply with only the code - no markdown fences, no explanation."
)

code_tests = [
    {"input": "1", "expected": "1"},
    {"input": "3", "expected": "14"},
    {"input": "10", "expected": "385"},
]


def strip_fences(text: str) -> str:
    """Remove markdown code fences if the policy ignores instructions."""
    return re.sub(r"^```(?:python)?\s*\n|\n?```\s*$", "", text.strip())


for i in range(3):
    candidate = strip_fences(simple_complete(CODING_TASK))
    score = verifier.verify(candidate, code_tests)
    print(f"candidate {i + 1}: reward = {score:.2f}")
    print("  " + candidate.replace("\n", "\n  ") + "\n")

candidate 1: reward = 1.00
  n = int(input())
  print(sum(i * i for i in range(1, n + 1)))

candidate 2: reward = 1.00
  n = int(input())
  print(sum(i*i for i in range(1, n+1)))

candidate 3: reward = 1.00
  n = int(input())
  print(sum(i*i for i in range(1, n+1)))



#### ❓ Question #4

The code verifier returns *fractional* rewards (fraction of tests passed) while the math verifier is binary. What are the benefits and risks of partial credit as a training signal? And concretely: what could a policy-generated program do to a verifier that runs candidates directly on the host, and which parts of that threat does our timeout **not** cover?

##### Answer:

**Benefits of partial credit.** A fractional reward is a *denser, smoother* signal. A program that passes 7/10 tests is meaningfully better than one that passes 2/10, and partial credit captures that gradient — whereas binary reward gives both a flat 0 and tells the policy nothing about how close it was. Denser reward means:
- more learnable structure early on (partial progress is rewarded, so the policy climbs rather than facing an all-or-nothing cliff),
- lower reward variance and less credit-assignment ambiguity, and
- a natural curriculum — the policy first learns to pass easy cases, then the hard/edge ones.

**Risks of partial credit.** The graded signal is only as trustworthy as the test suite, and partial credit *rewards partial solutions*, which is exactly what reward-hacking exploits:
- **Overfitting to the visible tests.** The policy can special-case the known inputs (`if n == 3: print(14)`) and collect credit without a general solution. Fractional credit makes this worse than binary because even partial hard-coding pays.
- **Reward is a biased proxy for correctness.** If your tests miss edge cases (n=0, negatives, large n), a program can score 1.0 while being wrong — and the policy will find and inhabit that gap. Mitigations: **hidden held-out tests**, adversarial/edge-case coverage, and not revealing which cases are checked.
- **Gaming the graded scale**, e.g., printing plausible outputs that coincidentally match on some cases.

**What malicious generated code could do to a host-run verifier.** We are running arbitrary model output with `subprocess.run([sys.executable, "-c", code], …)` on the host — that's remote-code-execution-by-design. A program could:
- **read or exfiltrate the filesystem** — dump `~/.ssh`, `.env`, API keys, the audit trail itself, and POST them out over the network;
- **open sockets / make network calls** — exfiltrate data, pull in a second-stage payload, or attack internal services;
- **write or delete files** — corrupt artifacts, plant malware, tamper with `verifier.jsonl` to hide its own tracks;
- **exhaust resources** — a fork bomb, memory balloon, or fill the disk;
- **poison the reward itself** — mutate environment/state so the verifier reports a pass it didn't earn.

**What the 5-second timeout does *not* cover.** The timeout only bounds *wall-clock duration of a single subprocess* — it kills long hangs/infinite loops. It does **nothing** about:
- **filesystem access** — reads/writes/deletes happen instantly, well under 5s;
- **network I/O / exfiltration** — a single fast request needs milliseconds;
- **fork bombs and spawned/backgrounded child processes** — `subprocess` with a timeout kills the *parent* it's waiting on but can leave detached children running after teardown;
- **memory/disk exhaustion** — a program can OOM the host or fill the disk long before 5s elapse;
- **fast destructive actions** in general — `rm -rf`, key theft, tampering with the audit log — all complete near-instantly.

The timeout is a *liveness* guard, not a *security boundary*. That's why the notebook flags this: in production the verifier must run in an isolated, disposable sandbox (gVisor, Firecracker/microVM, Vercel Sandbox, locked-down container) with no host filesystem, no network, and CPU/memory/pid limits — the execution environment *is* the security boundary, not the timeout.

## Task 7: Build Preference Pairs

Finally, we turn audited verifier output into training data. Within each group:

- **chosen** = verified-correct samples that were *not* flagged as hack-suspect,
- **rejected** = verified-incorrect samples,

and we take the cross product. The resulting `{prompt, chosen, rejected}` records are exactly the format [DPO-style trainers](https://huggingface.co/docs/trl/dpo_trainer) consume — while GRPO (Session 15) skips the pairing and uses the group rewards directly. Same verifier, two consumers.

Excluding flagged samples matters: a hack-suspect completion used as "chosen" would teach the next policy iteration to hack *more*.

In [11]:
def build_preferences(groups: list[list[Sample]], records: list[dict]) -> list[dict]:
    """Cross verified-correct (unflagged) winners with incorrect losers, per group."""
    flagged_responses = {r["response"] for r in records if r["suspected_hack"]}
    pairs = []
    for group in groups:
        winners = [s for s in group if s.verified_correct and s.response not in flagged_responses]
        losers = [s for s in group if not s.verified_correct]
        pairs.extend(
            {"prompt": winner.problem, "chosen": winner.response, "rejected": loser.response}
            for winner in winners
            for loser in losers
        )
    return pairs


pairs = build_preferences(groups, records)

PREFERENCES = Path("artifacts/preferences.jsonl")
with PREFERENCES.open("w") as f:
    for pair in pairs:
        f.write(json.dumps(pair) + "\n")

print(f"{len(pairs)} preference pairs -> {PREFERENCES}\n")
if pairs:
    example = pairs[0]
    print(f"prompt:   {example['prompt']}")
    print(f"chosen:   {example['chosen'][:120]}...")
    print(f"rejected: {example['rejected'][:120]}...")
else:
    print("No pairs this run - you need at least one correct AND one incorrect sample in the same group.")

3 preference pairs -> artifacts/preferences.jsonl

prompt:   If a train covers 60 km in 45 minutes, what is its speed in km/h?
chosen:   Given:
- Distance \( d = 60 \) km
- Time \( t = 45 \) minutes

**Step 1: Convert time to hours**

Since 1 hour = 60 minu...
rejected: Let's carefully go through the problem step by step.

1. **Identify the given information:**
   - Distance covered = 60 ...


#### 🏗️ Activity #1: Build Your Own Verifier

Math answers and unit tests are only two verifiable domains. Pick another — for example:

- **JSON schema conformance**: does the completion parse and validate against a schema?
- **SQL correctness**: does a generated query return the same rows as a reference query on a fixture database?
- **Regex/string transformation**: does the output match a deterministic expected transformation of the input?

Then, in the cell below:

1. Implement a reward function for your domain (binary or fractional — justify the choice).
2. Run the sample-and-verify loop over at least 3 prompts with `n_samples >= 3`.
3. Report the verified-correct rate, and note any hack-suspect behavior you observe (and how you'd detect it).

In [ ]:
"""Activity #1 — a JSON-schema-conformance verifier.

Domain: given a short natural-language spec that already contains the data,
the policy must emit a JSON object that (a) parses and (b) validates against a
target schema. This is verifiable: `json.loads` + `jsonschema` validation is a
deterministic checker, no human judgment or learned model in the loop.

Reward design — FRACTIONAL, justified:
    0.0  -> not parseable as JSON            ("not even JSON")
    0.5  -> parses but fails the schema       ("JSON, but wrong shape")
    1.0  -> parses AND validates              ("correct")
A graded signal gives the policy a curriculum (produce JSON -> produce *valid*
JSON) instead of an all-or-nothing cliff. The cost of partial credit is the
usual one (Question #4): a schema-valid object can still ignore the request, so
we pair the reward with a content hack-detector below.
"""

import json
from jsonschema import Draft202012Validator, FormatChecker

# `format` (e.g. "email") is advisory in JSON Schema and NOT enforced unless we
# attach a format checker - so we build validators with one, making the checker
# genuinely stricter rather than accepting any string for a formatted field.
FORMAT_CHECKER = FormatChecker()


def extract_json_text(response: str) -> str:
    """Best-effort extraction of the JSON payload from a completion.

    Deliberately robust to markdown fences of ANY language tag (the notebook's
    strip_fences only strips ```python) and to stray prose around the object, so
    the verifier doesn't punish otherwise-valid JSON for cosmetic wrapping - the
    same brittleness lesson as the boxed-answer extractor in Breakout Room #1.
    """
    text = response.strip()
    fenced = re.search(r"```[a-zA-Z0-9]*\s*\n?(.*?)```", text, re.DOTALL)
    if fenced:
        text = fenced.group(1).strip()
    if not text.startswith(("{", "[")):
        brace = re.search(r"\{.*\}", text, re.DOTALL)
        if brace:
            text = brace.group(0)
    return text


@dataclass
class SchemaProblem:
    spec: str                 # natural-language request (contains the data)
    schema: dict              # JSON Schema the output must satisfy
    must_contain: list[str]   # concrete values the output must actually use


PERSON_SCHEMA = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "age": {"type": "integer", "minimum": 0, "maximum": 120},
        "email": {"type": "string", "format": "email"},
    },
    "required": ["name", "age", "email"],
    "additionalProperties": False,
}

PRODUCT_SCHEMA = {
    "type": "object",
    "properties": {
        "id": {"type": "string"},
        "price": {"type": "number", "exclusiveMinimum": 0},
        "in_stock": {"type": "boolean"},
    },
    "required": ["id", "price", "in_stock"],
    "additionalProperties": False,
}

EVENT_SCHEMA = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "attendees": {"type": "array", "items": {"type": "string"}, "minItems": 1},
        "date": {"type": "string", "format": "date"},
    },
    "required": ["title", "attendees", "date"],
    "additionalProperties": False,
}

schema_problems = [
    SchemaProblem(
        "Represent this person as JSON: Ada Lovelace, age 36, email ada@example.com.",
        PERSON_SCHEMA,
        ["Ada Lovelace", "36", "ada@example.com"],
    ),
    SchemaProblem(
        "Represent this product as JSON: id SKU-42, price 19.99 dollars, currently in stock.",
        PRODUCT_SCHEMA,
        ["SKU-42", "19.99", "true"],
    ),
    SchemaProblem(
        "Represent this event as JSON: title 'Launch Party', attendees Sam and Priya, date 2026-08-01.",
        EVENT_SCHEMA,
        ["Launch Party", "Sam", "Priya", "2026-08-01"],
    ),
]


class JSONSchemaRewardFunction:
    """Graded verifiable reward: parses? -> validates against schema?"""

    def compute(self, response: str, schema: dict) -> float:
        obj = self._parse(response)
        if obj is None:
            return 0.0
        validator = Draft202012Validator(schema, format_checker=FORMAT_CHECKER)
        return 1.0 if validator.is_valid(obj) else 0.5

    @staticmethod
    def _parse(response: str):
        try:
            return json.loads(extract_json_text(response))
        except (json.JSONDecodeError, ValueError):
            return None


schema_reward_fn = JSONSchemaRewardFunction()

# Sanity checks against hand-written candidates - covering a ```json fence (the
# notebook's strip_fences would miss it), an out-of-range value, a bad email
# format (only caught because we attached a FormatChecker), and non-JSON.
assert schema_reward_fn.compute('{"name": "A", "age": 5, "email": "a@b.co"}', PERSON_SCHEMA) == 1.0
assert schema_reward_fn.compute('```json\n{"name": "A", "age": 5, "email": "a@b.co"}\n```', PERSON_SCHEMA) == 1.0
assert schema_reward_fn.compute('{"name": "A", "age": 500, "email": "a@b.co"}', PERSON_SCHEMA) == 0.5  # age out of range
assert schema_reward_fn.compute('{"name": "A", "age": 5, "email": "not-an-email"}', PERSON_SCHEMA) == 0.5  # bad format
assert schema_reward_fn.compute("not json at all", PERSON_SCHEMA) == 0.0
print("json-schema reward function OK")


@dataclass
class SchemaSample:
    spec: str
    response: str
    reward: float
    verified_correct: bool
    suspected_hack: bool


def looks_like_content_hack(response: str, problem: SchemaProblem, verified_correct: bool) -> bool:
    """Flag schema-valid output that ignores the request.

    A schema-valid object that omits the concrete values named in the spec is
    the JSON-domain analogue of a bare boxed guess: it satisfies the *checker*
    (structure) without addressing the *task* (content). We only flag samples
    the reward already accepted, since a rejected sample isn't a hack.
    """
    if not verified_correct:
        return False
    obj_text = extract_json_text(response).lower()
    return any(token.lower() not in obj_text for token in problem.must_contain)


# Confirm the detector separates an honest completion from a schema-valid one
# that ignores the spec.
_ada = '{"name": "Ada Lovelace", "age": 36, "email": "ada@example.com"}'
_junk = '{"name": "John Doe", "age": 30, "email": "x@y.co"}'
assert looks_like_content_hack(_ada, schema_problems[0], True) is False
assert looks_like_content_hack(_junk, schema_problems[0], True) is True
print("content-hack detector OK")


def sample_and_verify_schema(problem: SchemaProblem, n_samples: int = 4) -> list[SchemaSample]:
    """Sample a group of completions for one spec and verify each against the schema."""
    group = []
    for _ in range(n_samples):
        response = simple_complete(
            f"{problem.spec}\n\nReply with only the JSON object - no markdown fences, no explanation.",
            system="You output strictly valid JSON that conforms to the requested structure.",
        )
        reward = schema_reward_fn.compute(response, problem.schema)
        verified = reward == 1.0
        group.append(
            SchemaSample(
                spec=problem.spec,
                response=response,
                reward=reward,
                verified_correct=verified,
                suspected_hack=looks_like_content_hack(response, problem, verified),
            )
        )
    return group


schema_groups = [sample_and_verify_schema(p) for p in schema_problems]

total = sum(len(g) for g in schema_groups)
correct = sum(s.verified_correct for g in schema_groups for s in g)
flagged = sum(s.suspected_hack for g in schema_groups for s in g)

print(f"\nVerified-correct (schema-valid) rate: {correct / total:.0%} ({correct}/{total})")
print(f"Hack-suspect (valid but missing spec content): {flagged}\n")
for problem, group in zip(schema_problems, schema_groups):
    n_ok = sum(s.verified_correct for s in group)
    n_hack = sum(s.suspected_hack for s in group)
    rewards = ", ".join(f"{s.reward:.1f}" for s in group)
    print(f"  {n_ok}/{len(group)} valid  ({rewards})  hack:{n_hack}  {problem.spec[:55]}")

# Show one non-perfect or flagged sample, if any, to see what the verifier caught.
notable = [s for g in schema_groups for s in g if s.reward < 1.0 or s.suspected_hack]
if notable:
    s = notable[0]
    print(f"\n--- notable sample (reward {s.reward}, hack={s.suspected_hack}) ---")
    print(f"spec: {s.spec}")
    print(f"response: {s.response}")
else:
    print("\nAll samples were perfectly valid and used the spec content - "
          "raise temperature, shrink the model, or tighten the schema for more contrast.")

### Activity #1 — Notes & Report

**Domain & checker.** JSON-schema conformance. Each prompt gives a spec that already contains the data; the policy must emit a JSON object that parses (`json.loads`) *and* validates against a target `jsonschema`. Both steps are deterministic programs, so the reward is verifiable — no human judgment, no learned model.

**Why fractional reward.** I used a 3-level graded reward (`0.0` not-JSON → `0.5` JSON-but-wrong-shape → `1.0` valid). Like the code verifier's fraction-of-tests-passed, this is a *denser* signal than binary: it distinguishes "not even JSON" from "JSON with one wrong field," giving the policy a curriculum (first produce JSON, then produce *valid* JSON) instead of an all-or-nothing cliff. `verified_correct` still requires the full `1.0`, so only genuinely-valid samples become preference winners.

**Verified-correct rate.** Printed by the cell above as `correct / total` across 3 specs × 4 samples (`n_samples = 4 ≥ 3`). With `gpt-4.1-nano` on these simple schemas most samples validate; if you see 100%, tighten a schema (e.g., add a `pattern` or `format`) or raise temperature to recover the correct/incorrect contrast that training needs.

**Hack-suspect behavior & detection.** The reward only checks *structure*, so a policy can satisfy it while ignoring the *request* — emit a schema-valid object full of placeholder/hallucinated values (`{"name": "John Doe", "age": 30, ...}`) instead of the spec's data. This is the JSON analogue of the boxed-answer guess from Task 5. `looks_like_content_hack` flags any reward-accepted sample whose output is missing the concrete values named in the spec (`must_contain`). It only inspects samples the reward already passed, mirroring `looks_like_hack`.

Two more hacks worth hardening against here (same spirit as Question #3):
- **Schema-underfitting via loose schemas** — if the schema allowed `additionalProperties`, the policy could dump extra junk keys and still validate; I set `additionalProperties: False` and tight `required` lists to close that.
- **Type/format laxity** — e.g., an age of `"36"` (string) or a bogus email. The typed schema (`integer` with `minimum`/`maximum`, `format: email`) rejects those; production would add stricter `pattern`s and semantic checks (does the email domain resolve? is the date real?) as the checker, since format-valid ≠ correct.

**Where this plugs in.** `schema_reward_fn` is a drop-in reward for the GRPO loop, and unflagged `verified_correct` samples vs. `0.0`/`0.5` samples give the chosen/rejected contrast for DPO — the same two consumers as the math and code verifiers.

## Breakout Room #2 Summary

- Once you train against a verifier, it *is* the objective — Goodhart's law makes reward-hacking detection and an append-only audit trail (`artifacts/verifier.jsonl`) part of the core pipeline, not an afterthought.
- Code verification generalizes the idea: unit tests yield fractional rewards, and executing untrusted generated code demands sandboxing in anything beyond a demo.
- Verified groups become training data two ways: `{prompt, chosen, rejected}` pairs for DPO-style trainers, or raw group rewards for GRPO — with hack-suspect samples excluded so the next policy doesn't learn to cheat.

Where to go next: feed `artifacts/preferences.jsonl` to TRL's [`DPOTrainer`](https://huggingface.co/docs/trl/dpo_trainer); plug these reward functions into Session 15's GRPO run; or read how the labs do it at scale — [Tülu 3](https://arxiv.org/abs/2411.15124) (which coined RLVR) and [DeepSeek-R1](https://arxiv.org/abs/2501.12948).

---
## Conclusion: What We Built, Start to Finish

Walking back through the notebook, the full RLVR arc was:

1. **A policy** (Task 1) — a small API model sampled at temperature 1.0, chosen precisely because it is *sometimes wrong*.
2. **A verifiable domain** (Task 2) — math problems with ground-truth answers and a `\boxed{}` convention that makes checking mechanical.
3. **A reward function** (Task 3) — deterministic, normalized, and asymmetric (+1.0 / −0.1) so early failure doesn't teach refusal.
4. **The sampling loop** (Task 4) — *groups* of completions per prompt, the same structure GRPO computes advantages over, and the source of correct/incorrect contrast.
5. **Adversarial thinking** (Task 5) — once you train against a verifier it *is* the objective, so hack detection and an append-only audit trail are part of the pipeline, not an afterthought.
6. **A second domain** (Task 6) — code judged by unit tests, showing rewards can be fractional and that verification can mean *executing untrusted output*.
7. **Training data** (Task 7) — audited verifier decisions became `{prompt, chosen, rejected}` pairs, with hack-suspects excluded so the next policy iteration doesn't learn to cheat.

The single idea underneath all of it: **wherever a deterministic program can check correctness, you can turn cheap inference into training signal** — no human labelers, no learned reward model. The verifier's quality *is* the ceiling on the policy's quality.

## What Looks Different in Production

This notebook is the smallest honest version of RLVR. Scaling it into a real training pipeline changes nearly every component:

**Sandboxed code execution.** Our `CodeVerifier` runs model-generated code in a bare `subprocess` on your machine — fine for a demo, unacceptable in production, where the policy *will* eventually generate code that reads the filesystem, opens sockets, or fork-bombs the host (and our timeout catches none of that). Production verifiers execute candidates in isolated, disposable environments: [Vercel Sandbox](https://vercel.com/docs/vercel-sandbox) is a good example — ephemeral microVMs built to run untrusted, LLM-generated code with CPU/memory limits, network controls, and full teardown after each run. Self-hosted equivalents include gVisor, Firecracker microVMs, or locked-down containers. The rule: the verifier defines the reward, so the verifier's execution environment is a *security boundary*.

**Scale and throughput.** Five problems × 4 samples becomes tens of thousands of prompts × 8–64 samples per RL step. Sequential API calls give way to async/batched sampling against a dedicated inference fleet (vLLM, as in Session 15) — generation throughput, not the policy update, is usually the bottleneck.

**Verifier hardening.** Regex extraction and exact match get replaced by symbolic math checkers (e.g., SymPy-based equivalence), multiple test suites with hidden held-out cases, and stacked format rewards — because at scale, every blind spot *will* be found and exploited.

**Governance.** Our `verifier.jsonl` becomes real infrastructure: versioned datasets, per-run ledgers, flagged-sample review queues, and dashboards tracking reward distributions for drift. When someone asks "was this model trained on honest rewards?", the audit trail is the answer — this is exactly the artifact regulated environments (the origin of this material) require.

**The training loop itself.** The preference pairs here feed an actual policy update — DPO or GRPO — and then the loop *repeats* against the updated policy: sample, verify, update, again. One pass through this notebook is a single iteration of that flywheel.